# Camada Gold: Exportação para Power BI

Exportação idempotente das tabelas consolidadas da camada Gold para arquivos CSV no volume da Landing Zone, viabilizando a importação e visualização posterior no Power BI.

In [0]:
%run ./00_Config

In [0]:
# Importação das bibliotecas necessárias
import os

# Caminho de destino para exportação no volume da Landing Zone
export_path = f"{landing_path}/export_power_bi"

# Mapeamento das tabelas Gold a serem exportadas para CSV
tabelas_para_exportar = {
    "fato_chamados": f"{gold_schema}.fato_chamados",
    "agg_visao_executiva_mensal": f"{gold_schema}.agg_visao_executiva_mensal",
    "dim_canais": f"{gold_schema}.dim_canais",
    "dim_motivos": f"{gold_schema}.dim_motivos",
    "dim_atendentes": f"{gold_schema}.dim_atendentes",
    "dim_clientes": f"{gold_schema}.dim_clientes",
    "dim_data": f"{gold_schema}.dim_data"
}

# Execução da exportação idempotente em formato CSV com cabeçalho
for nome_arquivo, tabela_delta in tabelas_para_exportar.items():
    df = spark.table(tabela_delta)
    (
        df.coalesce(1)
        .write
        .format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .mode("overwrite")
        .save(f"{export_path}/{nome_arquivo}")
    )
    print(f"[OK] Tabela {tabela_delta} exportada com sucesso para {export_path}/{nome_arquivo}")